# Notebook 07 — Publication-Quality Figures

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 7 of 12  
**Time estimate:** 90 minutes

> Figures are the most-read part of any paper. A figure that is visually clear,
> colour-accessible, and correctly sized for the journal's column width communicates
> results faster than any paragraph can. This notebook covers matplotlib conventions
> for scientific publication.

---
## Step 1 — Motivation: What Separates a Publication Figure from a Draft Figure

A draft figure is for yourself: you understand the axes, the colours, and the
scale. A publication figure must be self-explanatory to a reader who has not
read your methods — using only the panel labels and the caption.

---
## Step 2 — Journal Specifications

| Journal | Single column | Double column | Max file size | DPI |
|---------|--------------|---------------|--------------|-----|
| Nature family | 89 mm | 183 mm | 10 MB | ≥300 |
| Cell family | 85 mm | 178 mm | 10 MB | ≥300 |
| PLOS journals | 83 mm | 171 mm | 10 MB | ≥300 |
| Nucleic Acids Research | 83 mm | 173 mm | 10 MB | ≥300 |

**At 300 DPI:** 89 mm = 1051 px. Always design in mm, export at ≥300 DPI.
**Font size:** minimum 7 pt after scaling to print size. If text < 7 pt in the
final printed column, the figure fails editorial review.

---
## Step 3 — Colour Accessibility

8% of men and 0.5% of women are colour-blind (predominantly red-green).
A figure that is only distinguishable by hue alone fails for this population.

**Rules:**
1. Use a perceptually uniform palette: Viridis, Plasma, Cividis (sequential);
   ColorBrewer palettes (categorical). Never use default MATLAB jet or Excel default colours.
2. Distinguish elements by shape + colour, not colour alone.
3. Test with a deuteranopia simulator: `matplotlib` → greyscale conversion is a quick test.

**Safe categorical palette (Tableau-10 subset):**  
`['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#76b7b2', '#edc948']`

**Safe sequential:** `viridis`, `plasma`, `cividis`.
**Safe diverging:** `RdBu_r`, `PuOr`, `BrBG`.

---
## Step 4 — Figure Anatomy

Every panel must have:
- **Panel label:** bold uppercase A, B, C... in the top-left corner (font size = 10–12 pt)
- **Axis labels:** with units in parentheses — "AUROC (0–1)" not just "AUROC"
- **Tick labels:** readable at final print size (≥7 pt)
- **Legend:** inside the panel if it doesn't obscure data
- **Error bars:** labelled in legend as SD, SEM, or 95% CI — never unlabelled
- **Statistical annotations:** * p<0.05, ** p<0.01, *** p<0.001, ns = p≥0.05

**Caption anatomy (standalone = self-explanatory):**
1. One sentence: what the figure shows overall
2. One sentence per panel: what it shows + the key number
3. Abbreviations defined the first time used in the caption
4. N and replicate information
5. Statistical test used

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

rng = np.random.default_rng(42)

# ---- Global rcParams for publication-quality figures ----
# Set these at the top of every publication notebook.

PUB_RCPARAMS = {
    # Font
    'font.family':       'sans-serif',
    'font.size':         9,            # body text
    'axes.titlesize':    10,
    'axes.labelsize':    9,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'legend.fontsize':   8,
    # Lines
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'lines.linewidth':   1.5,
    # Layout
    'figure.dpi':        150,          # screen; use 300+ for export
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'figure.autolayout': False,
    # Colour cycle — Tableau-10 accessible subset
    'axes.prop_cycle':   plt.cycler(color=['#4e79a7','#f28e2b','#59a14f',
                                            '#e15759','#76b7b2','#edc948']),
    # Spines
    'axes.spines.top':   False,
    'axes.spines.right': False,
}

plt.rcParams.update(PUB_RCPARAMS)
print('Publication rcParams applied.')
print(f'Font size: {plt.rcParams["font.size"]} pt')
print(f'Colour cycle: {[c["color"] for c in plt.rcParams["axes.prop_cycle"]]}')

In [ ]:
# ---- Generate synthetic data for demonstration ----
n_tfs = 40
auroc_db = rng.beta(9, 1, n_tfs)
auroc_pwm = rng.beta(7, 2, n_tfs)
auroc_rf  = rng.beta(8, 1.5, n_tfs)

# Time series: learning curves (5 replicates)
n_epochs = 50
base_curve = 0.5 + 0.4 * (1 - np.exp(-np.linspace(0, 5, n_epochs)))
curves_train = np.array([base_curve + rng.normal(0, 0.01, n_epochs) for _ in range(5)])
curves_val   = np.array([base_curve * 0.97 + rng.normal(0, 0.015, n_epochs) for _ in range(5)])

# ---- Figure: before and after comparison ----
# Left column = draft style; Right column = publication style

fig = plt.figure(figsize=(14, 10))
gs = GridSpec(2, 4, figure=fig, wspace=0.5, hspace=0.55)

# ---- DRAFT style (bad) ----
ax_d1 = fig.add_subplot(gs[0, :2])
ax_d1.set_title('DRAFT: Bar chart (problematic)', fontsize=9, color='tomato')
methods = ['DeepBind-v2', 'RandomForest', 'PWM']
means = [auroc_db.mean(), auroc_rf.mean(), auroc_pwm.mean()]
ax_d1.bar(methods, means)
ax_d1.set_ylabel('auroc')
# No error bars, no panel label, tiny axis labels — intentionally bad

ax_d2 = fig.add_subplot(gs[1, :2])
ax_d2.set_title('DRAFT: Learning curve (problematic)', fontsize=9, color='tomato')
for c in curves_train:
    ax_d2.plot(c, color='blue')
for c in curves_val:
    ax_d2.plot(c, color='green')
ax_d2.set_xlabel('epoch')
ax_d2.set_ylabel('auroc')
# Multiple colours with no legend, no replicate shading, no panel label

# ---- PUBLICATION style (good) ----
ax_p1 = fig.add_subplot(gs[0, 2:])
ax_p1.set_title('PUBLICATION: Correctly formatted', fontsize=9, color='steelblue')

colors = ['#4e79a7', '#59a14f', '#f28e2b']
for i, (method, vals, color) in enumerate(zip(methods, [auroc_db, auroc_rf, auroc_pwm], colors)):
    ax_p1.bar(i, vals.mean(), color=color, edgecolor='black', lw=0.6, alpha=0.85,
                yerr=vals.std(), capsize=4, error_kw={'linewidth': 1.2})
    # Strip plot overlay
    jitter = rng.uniform(-0.15, 0.15, len(vals))
    ax_p1.scatter(np.full(len(vals), i) + jitter, vals, color='black', s=6, zorder=3, alpha=0.5)

ax_p1.set_xticks([0, 1, 2]); ax_p1.set_xticklabels(methods)
ax_p1.set_ylabel('AUROC (0–1)')
ax_p1.set_ylim(0.5, 1.08)
ax_p1.text(-0.15, 1.10, 'A', transform=ax_p1.transAxes, fontsize=12, fontweight='bold')
ax_p1.text(0.5, 0.97, '***', ha='center', transform=ax_p1.get_xaxis_transform(), fontsize=12)
ax_p1.axhline(auroc_pwm.mean(), color='grey', ls=':', lw=0.8)

ax_p2 = fig.add_subplot(gs[1, 2:])
train_mean = curves_train.mean(axis=0)
train_std  = curves_train.std(axis=0)
val_mean   = curves_val.mean(axis=0)
val_std    = curves_val.std(axis=0)
epochs = np.arange(n_epochs)
ax_p2.plot(epochs, train_mean, color='#4e79a7', label='Train (n=5 replicates)')
ax_p2.fill_between(epochs, train_mean - train_std, train_mean + train_std,
                     color='#4e79a7', alpha=0.25)
ax_p2.plot(epochs, val_mean, color='#f28e2b', label='Validation')
ax_p2.fill_between(epochs, val_mean - val_std, val_mean + val_std,
                     color='#f28e2b', alpha=0.25)
ax_p2.set_xlabel('Training epoch')
ax_p2.set_ylabel('AUROC (0–1)')
ax_p2.legend(loc='lower right')
ax_p2.text(-0.15, 1.05, 'B', transform=ax_p2.transAxes, fontsize=12, fontweight='bold')

# Greyscale preview
fig_gs, axes_gs = plt.subplots(1, 2, figsize=(10, 4))
for i, (label, vals, marker) in enumerate(zip(methods, [auroc_db, auroc_rf, auroc_pwm],
                                                ['o', 's', '^'])):
    axes_gs[0].scatter([i]*len(vals) + rng.uniform(-0.1, 0.1, len(vals)), vals,
                        marker=marker, color=f'{0.2*i}', s=20, label=label)
axes_gs[0].set_xticks([0,1,2]); axes_gs[0].set_xticklabels(methods)
axes_gs[0].set_title('Greyscale + shape test\n(should be readable without colour)')
axes_gs[0].legend()
axes_gs[1].axis('off')
axes_gs[1].text(0.1, 0.7, 'If the figure is readable in greyscale\nand distinguishable by shape,\nit passes the colour-blindness test.',
                 transform=axes_gs[1].transAxes, fontsize=11,
                 bbox=dict(boxstyle='round', facecolor='lightyellow'))
plt.tight_layout()
plt.savefig('figures_greyscale_test.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(fig.number)
plt.savefig('pub_quality_figures.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Caption generator utility ----

def generate_caption(figure_title, panels):
    """
    Generate a draft figure caption.
    panels: list of (label, description, key_number) tuples
    """
    lines = [f'Figure X. {figure_title}.']
    for label, desc, key_num in panels:
        lines.append(f'({label}) {desc} {key_num}.')
    return '\n'.join(lines)

caption = generate_caption(
    figure_title='DeepBind-v2 outperforms baseline methods on the ENCODE TF benchmark',
    panels=[
        ('A', 'Comparison of AUROC across 40 TFs for three methods (bars = mean ± SD; dots = individual TFs).',
         'DeepBind-v2 achieves mean AUROC 0.92, outperforming PWM by 12 pp (*** p < 0.001, Wilcoxon signed-rank, n = 40)'),
        ('B', 'Training and validation learning curves for DeepBind-v2 (5 independent replicates; shaded region = ± 1 SD).',
         'Validation AUROC plateaus at epoch 30, with no evidence of overfitting'),
    ]
)
print('=== Draft figure caption ===')
print(caption)

---
## Step 8 — Exercises

See E07 in `exercises/README.md`: take any figure from Module 13 or 15,
apply the publication rcParams above, add panel labels and an error bar,
and write a two-sentence caption.

---
## Step 10 — Quiz

1. What is the minimum DPI for a publication figure? What column width does
   Nature use for single-column figures?
2. Why is the jet colourmap (rainbow) problematic for scientific figures?
   Name two safer alternatives for a diverging and a sequential dataset.
3. What five elements must every figure panel contain?
4. Write a one-sentence figure caption for a bar chart comparing two methods.
   Include: what it shows, the key result, n, and the statistical test.

---
## Step 12 — Reflection

> *[Apply the PUB_RCPARAMS dictionary above to a figure from any prior module
> notebook. Export it as a 300 DPI PNG. Then write a two-sentence caption.
> What changed between the draft and the publication version?]*

---
*Next: `08_open_science_practices.ipynb`*